# Retrieve Azure AI model catalog details

Look up a model from [Microsoft Foundry's public catalog](https://ai.azure.com/catalog/models/)
and return a Python dictionary containing its metadata and model card. This notebook makes
read-only requests. It does not deploy models, call inference, start a server, or publish anything.

**Audience:** Python developers exploring catalog metadata. **Prerequisites:** Python 3.10+
and internet access. The default path needs no Azure subscription or credentials.

1. Review the supported APIs and their coverage.
2. Retrieve the public catalog entry for one model and inspect its model card.
3. Optionally query Azure's authenticated APIs or save a local JSON copy.

Research and public-page checks: **2026-10-05**. Catalog contents change over time.


## Existing APIs and the chosen approach

| Interface | What it returns | Access and limits |
| --- | --- | --- |
| Public catalog detail page | Model card, publisher, tasks, token limits, modalities, license text, evaluations and catalog versions where supplied | Anonymous HTTPS. The embedded page-data format is **undocumented**, so this extraction can break when the website changes. |

The supported regional REST request is:

```http
GET https://management.azure.com/subscriptions/{subscriptionId}/providers/Microsoft.CognitiveServices/locations/{location}/models?api-version=2025-06-01
Authorization: Bearer <Entra token for https://management.azure.com/>
```

I did not find a documented anonymous API covering the complete content of the public model
catalog pages. The default below extracts the structured data already delivered with a model's
public HTML. It parses JSON only; it does not execute page JavaScript or call private backend APIs.
Prefer the supported SDK/REST examples below when their fields meet your needs.

The parser reads only the requested model queries, leaving unrelated publisher and benchmark
data alone. It rejects unexpected formats instead of silently returning incomplete HTML text.
This is direct data retrieval: no LLM or agent workflow is needed.


## Setup

From the `Administration/Model_Catalog` directory:

```bash
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
python -m pip install -r requirements.txt
cp .env.example .env      # Windows PowerShell: Copy-Item .env.example .env
jupyter lab output/jupyter-notebook/model_catalog_lookup.ipynb
```

Select the `.venv` Python kernel in your editor. If dependencies are missing from the active
kernel, run `%pip install requests python-dotenv` in a new cell and restart the kernel.
Edit `.env` to select the model and optional version. Use the exact catalog URL slug, such as
`Phi-4`, `gpt-4.1`, or `DeepSeek-R1`, rather than a deployment name or display label.


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

NOTEBOOK_PATH = Path("output/jupyter-notebook/model_catalog_lookup.ipynb")
PROJECT_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / NOTEBOOK_PATH).is_file()),
    None,
)
if PROJECT_DIR is None:
    raise RuntimeError("Open the notebook from inside Administration/Model_Catalog.")
load_dotenv(PROJECT_DIR / ".env")  # Before constructing any HTTP/Azure clients.

MODEL_NAME = os.getenv("CATALOG_MODEL_NAME", "Phi-4").strip()
MODEL_VERSION = os.getenv("CATALOG_MODEL_VERSION", "").strip() or None
print({"model": MODEL_NAME, "version": MODEL_VERSION or "catalog default"})

{'model': 'Phi-4', 'version': 'catalog default'}


## Decode the public page data

The current site uses a React Router reference table: object keys and values point into a JSON
array. These helpers decode just ordinary JSON objects, arrays and scalar values used by model
records. Unsupported types, broken references or a changed query shape raise `CatalogFormatError`.
Null and absent source values remain `None`; missing capabilities do not mean unsupported.


In [2]:
import json
import re
from datetime import datetime, timezone
from typing import Any
from urllib.parse import quote, urlsplit

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOG_BASE = "https://ai.azure.com/catalog/models/"


class CatalogFormatError(ValueError):
    """The public page no longer has the model-data shape we recognize."""


class ModelNotFoundError(LookupError):
    """The requested model/version was not returned by this source."""


def _field(table: list, row: dict, name: str) -> int:
    for key, ref in row.items():
        if re.fullmatch(r"_\d+", key) and int(key[1:]) < len(table):
            if table[int(key[1:])] == name:
                return ref
    raise CatalogFormatError(f"Missing page-data field: {name}")


def _decode(table: list, ref: int, ancestors: frozenset = frozenset()) -> Any:
    if type(ref) is not int:
        raise CatalogFormatError("Unsupported page-data reference type.")
    if ref in (-5, -7):  # turbo-stream null / undefined
        return None
    if not 0 <= ref < len(table) or ref in ancestors or len(ancestors) > 64:
        raise CatalogFormatError("Invalid, cyclic or excessively nested page data.")
    value = table[ref]
    ancestors = ancestors | {ref}
    if isinstance(value, dict):
        result = {}
        for key, child in value.items():
            if not re.fullmatch(r"_\d+", key):
                raise CatalogFormatError("Unsupported object key encoding.")
            decoded_key = _decode(table, int(key[1:]), ancestors)
            if not isinstance(decoded_key, str):
                raise CatalogFormatError("Expected a string object key.")
            result[decoded_key] = _decode(table, child, ancestors)
        return result
    if isinstance(value, list):
        return [_decode(table, child, ancestors) for child in value]
    return value

In [3]:
def _model_queries(html: str, model_name: str) -> dict[str, dict]:
    pattern = r'window\.__reactRouterContext\.streamController\.enqueue\(("(?:\\.|[^"\\])*")\)'
    chunks = re.findall(pattern, html)
    if not chunks:
        raise CatalogFormatError("No catalog page data found; check the URL or site layout.")
    try:
        stream = "".join(json.loads(chunk) for chunk in chunks)
        # Only the initial reference table is used, not later streamed instructions.
        table = json.loads(stream.splitlines()[0])
        if not isinstance(table, list):
            raise CatalogFormatError("Expected a page-data reference table.")
        queries = {}
        for row in table:
            if not isinstance(row, dict):
                continue
            try:
                query_ref = _field(table, row, "queryKey")
            except CatalogFormatError:
                continue
            key = _decode(table, query_ref)
            if not isinstance(key, list) or len(key) not in (2, 3):
                continue
            if key[0] != "model" or key[1] != model_name:
                continue
            if len(key) == 3 and key[2] != "versions":
                continue
            state = table[_field(table, row, "state")]
            payload = _decode(table, _field(table, state, "data"))
            if not isinstance(payload, dict) or not isinstance(payload.get("value"), list):
                raise CatalogFormatError("Expected a model query with a value array.")
            queries["default" if len(key) == 2 else "versions"] = payload
        if "default" not in queries:
            raise CatalogFormatError("Requested model query missing; URL or page schema changed.")
        return queries
    except (IndexError, KeyError, TypeError, AttributeError, json.JSONDecodeError) as exc:
        raise CatalogFormatError("Could not decode the catalog page-data schema.") from exc


def _model_identity(record: dict) -> tuple[str | None, str | None]:
    if not isinstance(record, dict):
        raise CatalogFormatError("Expected a model record object.")
    properties = record.get("properties") or {}
    if not isinstance(properties, dict):
        raise CatalogFormatError("Expected model properties to be an object.")
    name = record.get("entityObjectId") or properties.get("name")
    version = record.get("version")
    if version is None:
        version = properties.get("alphanumericVersion")
    return name, str(version) if version is not None else None

Normalize the fields without inferring missing facts. `systemCatalogData` supplies typed values;
older entries can fall back to tags. The complete selected source record is kept in `raw` for
inspection. Version selection uses exact strings: an omitted version means the page's default,
not the highest version number. A requested version must occur in the embedded results.


In [4]:
def _normalize(record: dict) -> dict:
    annotations = record.get("annotations") or {}
    tags = annotations.get("tags") or {}
    system = annotations.get("systemCatalogData") or {}

    def field(key, tag=None):
        value = system.get(key)
        return value if value is not None else tags.get(tag or key)

    def values(key, tag=None):
        value = field(key, tag)
        if isinstance(value, str):
            return [part.strip() for part in value.split(",") if part.strip()]
        return value

    def number(key):
        value = field(key)
        if type(value) is int:
            return value
        return int(value) if isinstance(value, str) and value.isdigit() else None

    name, version = _model_identity(record)
    return {
        "name": name,
        "display_name": field("displayName") or name,
        "version": version,
        "registry": record.get("entityResourceName"),
        "asset_id": record.get("assetId"),
        "publisher": field("publisher", "author"),
        "summary": field("summary"),
        "context_window_tokens": number("textContextWindow"),
        "max_input_tokens": number("maxInputTokens"),
        "max_output_tokens": number("maxOutputTokens"),
        "input_modalities": values("inputModalities"),
        "output_modalities": values("outputModalities"),
        "tasks": values("inferenceTasks", "task"),
        "capabilities": values("modelCapabilities"),
        "deployment_types": values("deploymentTypes"),
        "deployment_options": values("deploymentOptions"),
        "languages": values("languages"),
        "license": field("license"),
        "lifecycle": field("lifecycle"),
        "model_card_markdown": annotations.get("description"),
        "license_text": tags.get("licenseDescription"),
        "evaluation_markdown": tags.get("evaluation"),
        "notes_markdown": tags.get("notes"),
        "raw": record,
    }


def parse_model_page(html: str, model_name: str, version: str | None = None) -> dict:
    queries = _model_queries(html, model_name)
    defaults = queries["default"]["value"]
    version_page = queries.get("versions", {})
    versions = version_page.get("value", [])
    candidates = versions + defaults if version is not None else defaults
    matches = [
        record
        for record in candidates
        if _model_identity(record)[0] == model_name
        and (version is None or _model_identity(record)[1] == str(version))
    ]
    unique = {record.get("assetId") or str(_model_identity(record)): record for record in matches}
    if not unique:
        raise ModelNotFoundError(
            f"{model_name!r}, version {version!r}, is absent from this page's results. "
            "Check the exact slug/version; the public page may omit some versions."
        )
    if len(unique) != 1:
        raise CatalogFormatError("Multiple model assets match; use the registry SDK explicitly.")
    result = _normalize(next(iter(unique.values())))
    result["available_versions"] = list(
        dict.fromkeys(
            v
            for record in versions + defaults
            for n, v in [_model_identity(record)]
            if n == model_name and v is not None
        )
    )
    result["versions_complete"] = bool(
        version_page
        and version_page.get("totalCount") == len(versions)
        and not version_page.get("continuationToken")
        and version_page.get("nextSkip") is None
        and not version_page.get("shardErrors")
        and not version_page.get("numberOfEntityContainersNotQueried")
    )
    result["source_kind"] = "public_catalog_page_undocumented_format"
    return result

## Retrieve a given model

`get_model_details(name, version=None)` requests only that model's public detail page.
HTTP timeouts are bounded; a transient connection/502/503/504 failure is retried up to twice.
HTTP 429 is surfaced so you can wait before retrying. Requests are not sent to arbitrary URLs.
No page data is cached or written to disk automatically.


In [5]:
def get_model_details(model_name: str, version: str | None = None) -> dict:
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,199}", model_name):
        raise ValueError("Use a catalog model slug, not a URL, path or display label.")
    source_url = CATALOG_BASE + quote(model_name, safe="")
    retry = Retry(
        total=2,
        backoff_factor=1,
        status_forcelist=[502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=False,
    )
    with requests.Session() as session:
        session.mount("https://", HTTPAdapter(max_retries=retry))
        response = session.get(
            source_url,
            timeout=(10, 60),
            allow_redirects=False,
            headers={"User-Agent": "ModelCatalogNotebook/0.1", "Accept": "text/html"},
        )
        if response.status_code == 404:
            raise ModelNotFoundError(f"No public catalog page for {model_name!r}.")
        response.raise_for_status()
        if response.is_redirect or "text/html" not in response.headers.get("Content-Type", ""):
            raise CatalogFormatError(
                "Expected a public HTML detail page, got a redirect or other content."
            )
        result = parse_model_page(response.text, model_name, version)
        result["source_url"] = source_url
        result["retrieved_at_utc"] = datetime.now(timezone.utc).isoformat()
        return result

In [6]:
model = get_model_details(MODEL_NAME, MODEL_VERSION)
summary_fields = [
    "name",
    "version",
    "registry",
    "publisher",
    "summary",
    "context_window_tokens",
    "max_output_tokens",
    "input_modalities",
    "output_modalities",
    "tasks",
    "capabilities",
    "deployment_types",
    "license",
    "lifecycle",
    "available_versions",
    "versions_complete",
    "source_url",
    "retrieved_at_utc",
]
print(json.dumps({key: model[key] for key in summary_fields}, indent=2, ensure_ascii=False))

{
  "name": "Phi-4",
  "version": "8",
  "registry": "azureml",
  "publisher": "Microsoft",
  "summary": "Phi-4 14B, a highly capable model for low latency scenarios.",
  "context_window_tokens": 16384,
  "max_output_tokens": 16384,
  "input_modalities": [
    "text"
  ],
  "output_modalities": [
    "text"
  ],
  "tasks": [
    "chat-completion"
  ],
  "capabilities": null,
  "deployment_types": [
    "maap-inference",
    "maas-inference"
  ],
  "license": "MIT",
  "lifecycle": null,
  "available_versions": [
    "6",
    "4",
    "8",
    "7",
    "5",
    "3",
    "1",
    "2"
  ],
  "versions_complete": true,
  "source_url": "https://ai.azure.com/catalog/models/Phi-4",
  "retrieved_at_utc": "2026-10-05T17:40:42.438184+00:00"
}


The model card and other text are kept in full, even when a preview is shortened. Source fields
can disagree (for example, prose and typed token limits); inspect `model["raw"]` before making
deployment decisions. Registry replication regions are not deployment regions, so they are not
presented as availability. Pricing prose/links, where present in the card, are not a price quote.


In [7]:
# Plain text previews keep remote Markdown/HTML from being executed or rendered.
for field_name in ("model_card_markdown", "license_text", "evaluation_markdown", "notes_markdown"):
    content = model[field_name]
    print(f"\n{field_name}: {len(content or ''):,} characters")
    print(content[:1200] if content else "Not supplied by this catalog record.")
    if content and len(content) > 1200:
        print("[Preview truncated; the dictionary contains the complete text.]")


model_card_markdown: 1,679 characters
Phi-4 is a state-of-the-art open model built upon a blend of synthetic datasets, data from filtered public domain websites, and acquired academic books and Q&A datasets. The goal of this approach was to ensure that small capable models were trained with data focused on high quality and advanced reasoning.

Phi-4 underwent a rigorous enhancement and alignment process, incorporating both supervised fine-tuning and direct preference optimization to ensure precise instruction adherence and robust safety measures.

For more information, reference the [Phi-4 Technical Report](https://www.microsoft.com/en-us/research/uploads/prod/2024/12/P4TechReport.pdf).

### Model Architecture

Phi-4 is a 14B parameters, dense decoder-only transformer model. 

### Training Data

Our training data is an extension of the data used for Phi-3 and includes a wide variety of sources from:

1. Publicly available documents filtered rigorously for quality, selected high-qualit

## Troubleshooting and next steps

- **404 / ModelNotFoundError:** use the exact URL slug and a version actually returned by that
  page. If `versions_complete` is false, the embedded version results are incomplete. Use the
  registry SDK to enumerate versions.
- **CatalogFormatError:** the site changed or returned another page. Inspect the public page
  and update the parser; a blank model card should not be substituted for a parsing failure.
- **401 / 403:** for optional Azure calls, verify your Entra identity, tenant and read access.
- **429 / timeout:** wait before trying again. Keep this notebook to individual lookups.
- **Null fields:** the source did not supply that value. Do not interpret this as a model limit
  of zero, no capability support, or a guarantee about deployment availability.

The default public path was checked with live Phi-4 and gpt-4.1 pages. Offline tests cover model
selection, metadata normalization, schema failures, HTTP errors and ARM pagination. Authenticated
SDK/REST integration checks remain a follow-up for an environment with Azure access.

If this later becomes a service, add schema monitoring, caching, rate limiting, authentication,
Azure Monitor/Application Insights telemetry, and CI integration checks. Prefer managed identity
and Key Vault for any future secrets. Consider VNet/network rules and scaling only at that stage.
Nothing in this notebook publishes or deploys a service.
